# Phase 2 - oracle@N coverage gate (no retrain)

Fresh Colab, **A100 + High-RAM recommended**, **Run All**. Reloads the P6 two-stage adapter and, for a
slice of the holdout, draws N samples/row and reports **oracle@k** (best achievable by a perfect
reranker) and **best_of_N** (the canonical reranker's pick). This is the gate that decides the next move
(docs/collapse_fix/README.md):
- oracle/best_of_N **>= ~0.6** -> coverage good -> do best-of-N (Doc 02) + self-distillation (Doc 03)
- **0.3-0.6** -> ship Doc 02 as a quality knob; Doc 03 ceiling capped
- **<= ~0.3** -> escalate to sequence-level RL (reserve)

Cost: LIMIT×N sampled 64-token trajectories, batched (num_return_sequences), 2 temperatures -> tens of
minutes on A100 at the defaults. The one-time ~9.85GB corpus stage dominates a fresh VM.


In [ ]:
# CELL 1 - provision the runtime (clone, install, HF token, stage the corpus)
import os, pathlib, subprocess, sys
REPO = '/content/SLM'
BRANCH = 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = _env_token('HF_TOKEN') or getpass.getpass('HF_TOKEN (read, for staging + adapter): ').strip()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')


In [ ]:
# CELL 2 - build base_resized + download the P6 two-stage adapter from HF
import os, pathlib, subprocess, sys, json
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
D = pathlib.Path('models/base_resized')
if not ((D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file()):
    print('building base_resized (loads base in fp32 on CPU ~12GB; A100/High-RAM recommended) ...')
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
vr = json.load(open(D / 'vocab_resize_manifest.json'))
assert vr.get('tokenizer_version') == 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f', vr.get('tokenizer_version')
print('base_resized OK; identity:', vr['tokenizer_version'])

from huggingface_hub import snapshot_download
REPO_ID = 'ericrcwu/LUT_SLM_sft_adapters'
ADAPTER_SUBFOLDER = 'p6_twostage_d0f9c744_smokefull'
snapshot_download(repo_id=REPO_ID, allow_patterns=[ADAPTER_SUBFOLDER + '/*'],
                  local_dir='models/sft_adapters', token=os.environ.get('HF_TOKEN'))
ADAPTER = 'models/sft_adapters/' + ADAPTER_SUBFOLDER
assert pathlib.Path(ADAPTER, 'adapter_model.safetensors').is_file(), ADAPTER
print('adapter ready:', ADAPTER)


In [ ]:
# CELL 3 - run the oracle@N gate (greedy baseline was 0.159; real-code ceiling ~0.89)
import os, sys, json, subprocess
ADAPTER = 'models/sft_adapters/p6_twostage_d0f9c744_smokefull'
LIMIT, N = 32, 32          # LIMIT=0 -> full 120-row holdout; raise N for a higher ceiling (slower)

env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'eval.oracle_at_n',
       '--config', 'configs/candidate_two_stage.json',   # input_field=attribute_spec_text (matches P6)
       '--resized-model', 'models/base_resized', '--adapter', ADAPTER,
       '--limit', str(LIMIT), '--n', str(N), '--temperatures', '0.7,1.0', '--top-p', '0.9']
print('running:', ' '.join(cmd), '\n')
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(p.stdout[-9000:])
if p.returncode != 0:
    print('--- STDERR tail ---\n', p.stderr[-3000:])

def fmt(x):
    return 'n/a' if x is None else ('%.3f' % x)
for line in p.stdout.splitlines():
    if line.strip().startswith('{') and 'oracle_summary' in line:
        s = json.loads(line)['oracle_summary']
        print('\n================= ORACLE SUMMARY =================')
        print('adapter:', s['adapter'], '| n:', s['n'], '| limit:', s['limit'])
        for t, r in s['by_temperature'].items():
            curve = '  '.join('@%s=%s' % (k, fmt(r.get('oracle@%s' % k))) for k in (1, 4, 8, 16, 32))
            print('t=%s  %s  best_of_N=%s  pick_collapse=%s' % (
                t, curve, fmt(r.get('best_of_N')), fmt(r.get('best_pick_collapse_rate'))))
print('\nGate (provisional): best_of_N >=0.6 GOOD (do 02+03) | 0.3-0.6 CAPPED | <=0.3 RL.')
print('See docs/collapse_fix/README.md. Paste the ORACLE SUMMARY + the [oracle][gate] line back.')
